In [1]:
from langchain_ollama import OllamaEmbeddings

# 创建向量模型,我们今天使用ollama
ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

# 初始化向量数据库客户端对象
from pymilvus import MilvusClient
from app.core.config import settings

# 初始化向量数据库
milvus_client = MilvusClient(uri=settings.rag.milvus_url)

# Collection名称:集合,指的就是表名字
collection_name = "my_collection_1"

In [4]:
import json
# 从向量数据库中导入向量匹配类
from pymilvus import AnnSearchRequest
# 定义混合检索函数
def hybrid_search(user_question,ranker):
    """
    创建两个查询对象,将结果混合在一起重新进行权重计算,再排序
    :param user_question: 用户提示词
    :param ranker: 每种检索的排序权重
    :return: 返回新的权重计算后的匹配结果
    """
    # 用户提示词转稠密向量
    user_vector = ollama_embeddings.embed_query(user_question)

    # 创建稠密匹配对象
    dense_query = AnnSearchRequest(
        # 不需要输入集合名,设置匹配参数
        data=[user_vector],
        # 匹配稠密向量字段
        anns_field='dense',
        # 查询使用余弦相似度算法
        param={"metric_type": "COSINE"},
        # 最大匹配三条
        limit=3
    )

    # 创建稀疏向量匹配对象
    sparse_query = AnnSearchRequest(
        data=[user_question],
        # 匹配稠密向量字段
        anns_field='sparse',
        param={"metric_type": "BM25"},
        # 最大匹配三条
        limit=3
    )

    # 通过数据库客户端查询
    results = milvus_client.hybrid_search(
        # 使用混合检索方法
        collection_name=collection_name,
        # 组和两种请求
        reqs=[dense_query,sparse_query],
        # 设置返回条数
        limit=3,
        # 设置返回的字段
        output_fields=['id', 'h2', 'content'],
        # 设置每种请求的权重
        ranker=ranker
    )

    for hits in results:
        for hit in hits:
            print(json.dumps(hit,indent=2,ensure_ascii=False))

In [5]:
from pymilvus import WeightedRanker
# 定义权重
ranker =  WeightedRanker(0.5,0.5)

# 调用函数进行匹配
hybrid_search('教育的负面作用是什么',ranker)



{
  "id": 21,
  "distance": 0.7263909578323364,
  "entity": {
    "h2": "第二节 教育和社会的关系",
    "content": "### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。",
    "id": 21
  }
}
{
  "id": 15,
  "distance": 0.4075165390968323,
  "entity": {
    "h2": "第一节 教育的功能",
    "content": "# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。",
    "id": 15
  }
}
{
  "id": 16,
  "distance": 0.39741578698158264,
  "entity": {
    "h2": "第二节 教育和社会的关系",
    "content": "## 第二节 教育和社会的关系  \n### （一）政治与教育的关系  \n政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  \n教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。",
    "id": 16
  }
}


In [6]:
# 定义权重
ranker =  WeightedRanker(0.5,0.5)

# 调用函数进行匹配
hybrid_search('教育漫画是谁写的',ranker)


{
  "id": 7,
  "distance": 0.35669344663619995,
  "entity": {
    "content": "### （七）教育学创立时期代表人物及主要思想  \n**昆体良**：西方第一个专门论述教育问题的教育家。  \n**培根**：在《论科学的价值和发展》中首次把“教育学”作为一门独立的科学确立下来。  \n**夸美纽斯**（教育学之父、教育史上哥白尼）：代表作《大教学论》是教育学成为一门独立学科的标志。提出“泛智教育”、“班级授课制”和“学年制”；把教师比喻为太阳底下最光辉的事业。  \n**康德**：教育学作为一门课程在大学里讲授，始于康德。  \n**赫尔巴特**（传统教育代表、科学教育学之父）：《普通教育学》是教育学作为一门规范、独立的学科正式诞生的标志。提出以伦理学和心理学作为教育学的基础；教学要有教育性；三中心论：教师、教材、课堂；四阶段论：清楚、联想、系统和方法。  \n**杜威**（现代教育代表、实用主义哲学之父、儿童中心主义论、教育无目的论）：《民主主义与教育》提出教育的本质：教育即生活、教育即生长、教育即经验的改组或改造；学校即社会；从做中学；连带学习；三中心论：儿童、经验、活动；五步教学法：设疑—分析—假设—推断—验证。  \n**卢梭**：《爱弥儿》，提倡自然主义教育思想，认为教育的任务应该使儿童归于自然。  \n**洛克**：《教育漫画》，提出白板说，倡导绅士教育。  \n**裴斯泰洛奇**：《林哈德与葛笃德》，教育心理学化。  \n**斯宾塞**：反对思辨，主张用实证方法研究知识价值；生活预备说；科学知识最有价值，制定以科学知识为核心的课程体系。  \n**陶行知**：提出生活教育理论：生活即教育，社会即学校，教学做合一；“捧着一颗心来”（师德）；伟大的人民教育家。  \n**蔡元培**：思想自由，兼容并包；以美育代宗教；学界泰斗，人世楷模。",
    "h2": "第一节 中外教育家及其教育思想",
    "id": 7
  }
}
{
  "id": 4,
  "distance": 0.29790112376213074,
  "entity": {
    "content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”